# AI Powered

This notebook demonstrates AI-assisted semantic model workflows, such as extracting SQL lineage, generating measure descriptions, and organizing measure folders.

In [ ]:
%pip install -q openai 2>/dev/null

In [ ]:
# Parameters
workspace_name = "Demo"
dataset_name = "card_udf"

In [ ]:
# Imports
# Imports used by demo(s): AI; General; Put Measures in Organized Folders
import json
# Imports used by demo(s): AI; General; Put Measures in Organized Folders
import pandas as pd
# Imports used by demo(s): AI; Display what measure descriptions would be; Write measure descriptions; General; Put Measures in Organized Folders
import sempy_labs as labs
# Imports used by demo(s): AI; General; Put Measures in Organized Folders
import synapse.ml.aifunc as aifunc

## Real world skills - Parsing SQL Tables Used
##### AI

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model

dataset = dataset_name
workspace = workspace_name

# Pull M Code
rows = []
with labs.tom.connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
    for t in tom.model.Tables:
        p = next((x for x in t.Partitions), None)
        if p is None:
            continue
        m_code = getattr(getattr(p, "Source", None), "Expression", None)
        if isinstance(m_code, str) and m_code.strip():
            rows.append({"PowerBI_Table_Name": t.Name, "M_Code": m_code})

# One prompt + one AI call
prompt = (
    "Return ONE JSON OBJECT with key 'rows'. "
    "rows must be an array of objects with: PowerBI_Table_Name, Criteria_Type, Schema_Database_Table. "
    "Criteria_Type allowed: FROM, JOIN, LEFT JOIN, RIGHT JOIN, INNER JOIN, FULL JOIN, FULL OUTER JOIN, CROSS JOIN, ITEM. "
    "Parse SQL refs from each M_Code.\n"
    + json.dumps(rows)
)

raw = pd.Series([prompt]).ai.generate_response(
    "Extract table references from M code.",
    response_format="json_object"
).iloc[0]

# Parse to dataframe
obj = json.loads(raw) if isinstance(raw, str) else raw
df_sql_refs = pd.DataFrame(obj.get("rows", [])).drop_duplicates().reset_index(drop=True)
df_sql_refs

## Real world skills - AI Measure Description

##### Display what measure descriptions would be

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.generate_measure_descriptions

with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=True
) as tom:
    display(tom.generate_measure_descriptions())

##### Write measure descriptions

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.generate_measure_descriptions

# Apply descriptions to the model (readonly=False)
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=False
) as tom:
    display(tom.generate_measure_descriptions())

##### Write measure descriptions with Custom AI

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.all_measures
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.update_measure

dataset = dataset_name
workspace = workspace_name

custom_prompt = (
    "Write clear, business-friendly measure descriptions. "
    "Each description must be 1 sentence, start with 'Calculates', "
    "and avoid technical jargon when possible."
)

# Pull measures
rows = []
with labs.tom.connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
    for m in tom.all_measures():
        rows.append(
            {
                "Measure_Name": m.Name,
                "Table_Name": m.Parent.Name,
                "Expression": m.Expression,
            }
        )

# One prompt + one AI call
prompt = (
    "Return ONE JSON OBJECT with key 'rows'. "
    "rows must be an array of objects with: Measure_Name, Description. "
    f"Custom instructions: {custom_prompt} "
    "Do not include any extra text.\n"
    + json.dumps(rows)
)

raw = pd.Series([prompt]).ai.generate_response(
    "Generate measure descriptions.",
    response_format="json_object",
).iloc[0]

# Parse to dataframe
obj = json.loads(raw) if isinstance(raw, str) else raw
df_measure_descriptions = (
    pd.DataFrame(obj.get("rows", []))
    .drop_duplicates(subset=["Measure_Name"])
    .reset_index(drop=True)
)

# Apply descriptions
with labs.tom.connect_semantic_model(dataset=dataset, workspace=workspace, readonly=False) as tom:
    for _, row in df_measure_descriptions.iterrows():
        tom.update_measure(
            measure_name=row["Measure_Name"],
            description=row["Description"],
        )

df_measure_descriptions

## Real world skills - AI Measure Folder Organization

##### Put Measures in Organized Folders

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.all_measures
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.update_measure

dataset = dataset_name
workspace = workspace_name

# Pull measures
rows = []
with labs.tom.connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
    for m in tom.all_measures():
        rows.append(
            {
                "Measure_Name": m.Name,
                "Table_Name": m.Parent.Name,
                "Expression": m.Expression,
            }
        )

# One prompt + one AI call
prompt = (
    "Return ONE JSON OBJECT with key 'rows'. "
    "rows must be an array of objects with: Measure_Name, Display_Folder. "
    "Assign each measure to a short business-friendly display folder based on its name, table, and expression. "
    "Reuse the same folder names where possible. "
    "Do not include any extra text.\n"
    + json.dumps(rows)
)

raw = pd.Series([prompt]).ai.generate_response(
    "Assign display folders to measures.",
    response_format="json_object",
).iloc[0]

# Parse to dataframe
obj = json.loads(raw) if isinstance(raw, str) else raw
df_measure_folders = (
    pd.DataFrame(obj.get("rows", []))
    .drop_duplicates()
    .reset_index(drop=True)
)

# Apply folders
with labs.tom.connect_semantic_model(dataset=dataset, workspace=workspace, readonly=False) as tom:
    for _, row in df_measure_folders.iterrows():
        tom.update_measure(
            measure_name=row["Measure_Name"],
            display_folder=row["Display_Folder"],
        )

df_measure_folders